# Rare Galaxies in SGA-2025: Starter Notebook

This notebook is the starting point for the project described in
[`rare-galaxies-plan.md`](rare-galaxies-plan.md). It covers the Phase 1 setup and the
first Phase 2 experiments:

1. Load the embeddings and the catalog
2. Define the working sample
3. Look at nearest neighbors
4. Search with several query galaxies at once
5. Test what the embeddings encode
6. Rank outliers
7. Record visual inspection

Shared functions live in [`rare.py`](rare.py), next to this notebook. When you write a
function you expect to reuse, put it there. For background on PCA, UMAP, and cosine
similarity, see `doc/tutorials/SGA-ssl.ipynb`.

Cells marked **TODO** are yours to fill in. Once a section grows, move it to its own
numbered notebook.

## 0. Setup

Run this notebook with the SGAML kernel on NERSC JupyterHub.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from SGA.ssl import load_ssl_embeddings
from SGA.qa import notebook_style

import rare

notebook_style()

In [ ]:
REGION  = 'dr11-south'
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'
OUTDIR  = os.path.join(os.environ['SCRATCH'], 'rare-galaxies')  # cached results
os.makedirs(OUTDIR, exist_ok=True)

seed = 42
rng = np.random.default_rng(seed)

## 1. Load the data

`load_ssl_embeddings` joins the embeddings to the SGA-2025 catalog on `SGAID`. Each row
is one galaxy, with all of the catalog columns plus `embeddings` (2048-d) and
`projections` (128-d).

In [ ]:
data = load_ssl_embeddings(REGION, SSL_DIR)
print(f'{len(data):,d} galaxies')
print(data.colnames)

In [ ]:
cutout_index = rare.build_cutout_index(SSL_DIR, REGION)  # takes ~30 seconds
print(f'{len(cutout_index):,d} cutouts indexed')

## 2. Define the working sample

Two cuts keep the first analysis clean:

- **Full *grz* coverage.** Cutouts with a missing band are zero-filled, so they look
  unusual for uninteresting reasons.
- **Isolated galaxies.** Each cutout shows the whole galaxy *group*, so we keep groups
  with one member (`GROUP_MULT == 1`).

In [ ]:
has_grz  = np.array([all(b in str(bands) for b in 'grz') for bands in data['BANDS']])
isolated = np.asarray(data['GROUP_MULT']) == 1
keep = has_grz & isolated

print(f'grz coverage: {has_grz.sum():,d}')
print(f'isolated:     {isolated.sum():,d}')
print(f'both:         {keep.sum():,d} / {len(data):,d}')

sample = data[keep]

In [ ]:
# Unit-length vectors, so that a dot product is a cosine similarity.
proj = rare.normalize(sample['projections'])   # (N, 128)
emb  = rare.normalize(sample['embeddings'])    # (N, 2048)

sgaid  = np.asarray(sample['SGAID'])
row_of = {int(s): i for i, s in enumerate(sgaid)}  # SGAID -> row in `sample`

In [ ]:
# A random set of galaxies, to get oriented.
I = rng.choice(len(sample), size=25, replace=False)
rare.show_cutout_grid(sgaid[I], cutout_index)

## 3. Nearest neighbors of one galaxy

Pick a galaxy and show the galaxies closest to it in embedding space.

In [ ]:
# Find a galaxy by name with, e.g., sample['SGAID'][sample['OBJNAME'] == 'NGC 1068']
query = int(sgaid[0])          # <- change this
vecs  = proj                   # <- try emb (2048-d) instead

order, score = rare.rank_by_similarity(vecs, [row_of[query]])

rare.show_cutout_grid([query], cutout_index, ncols=1, figsize_per=3)
rare.show_cutout_grid(sgaid[order[:20]], cutout_index,
                      titles=[f'{s:.3f}' for s in score[order[:20]]])

In [ ]:
rare.print_viewer_links([query] + list(sgaid[order[:20]]), sample, REGION)

**TODO**

1. Repeat for about ten galaxies of different types. Do the neighbors look alike?
2. Compare `proj` with `emb`. Which gives more convincing neighbors?
3. Find five to ten ring galaxies by eye (or from the literature) and note their SGAIDs
   for the next section.

## 4. Search with several queries

With several examples of the same class, rank every galaxy by its *mean* similarity to
the examples. This is the simplest version of the ring search in the plan.

In [ ]:
query_sgaids = [int(s) for s in sgaid[:5]]   # <- TODO: replace with known ring galaxies

query_rows = [row_of[s] for s in query_sgaids]
order, score = rare.rank_by_similarity(vecs, query_rows)

rare.show_cutout_grid(query_sgaids, cutout_index)
rare.show_cutout_grid(sgaid[order[:50]], cutout_index, ncols=10, figsize_per=1.5)

Once we have a reference catalog of known rings, we can measure how well the search
works. Hold some known rings out of the query set, then ask how many candidates we must
inspect to recover them.

**TODO:** cross-match `sample` to a ring catalog (by RA, Dec) and build `is_ring`, a
boolean array with one entry per row of `sample`. Then run the cell below.

In [ ]:
is_ring = np.zeros(len(sample), dtype=bool)   # <- TODO: fill from a ring catalog

if is_ring.sum() > 20:
    ring_rows = rng.permutation(np.flatnonzero(is_ring))
    query_rows, heldout_rows = ring_rows[:10], ring_rows[10:]

    is_heldout = np.zeros(len(sample), dtype=bool)
    is_heldout[heldout_rows] = True

    order, score = rare.rank_by_similarity(vecs, query_rows)
    n, recall = rare.recall_curve(order, is_heldout)
    n_rand, recall_rand = rare.recall_curve(rng.permutation(order), is_heldout)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(n, recall, label='similarity search')
    ax.plot(n_rand, recall_rand, ls='--', color='gray', label='random order')
    ax.set_xscale('log')
    ax.set_xlabel('Number of candidates inspected')
    ax.set_ylabel('Fraction of held-out rings recovered')
    ax.legend()

## 5. What do the embeddings encode?

If a simple linear model can predict a catalog property from the embedding, that
property influences which galaxies count as "similar." We test angular size and axis
ratio here. R² near 1 means the property is strongly encoded; near 0 means it is not.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

I = rng.choice(len(sample), size=min(50_000, len(sample)), replace=False)

targets = {
    'log10 D26': np.log10(np.asarray(sample['D26'][I], dtype=float)),
    'b/a':       np.asarray(sample['BA'][I], dtype=float),
    # TODO: add magnitude, color, and redshift (check data.colnames)
}

for name, y in targets.items():
    good = np.isfinite(y)
    Xtrain, Xtest, ytrain, ytest = train_test_split(
        emb[I][good], y[good], test_size=0.25, random_state=seed)
    model = Ridge(alpha=1.0).fit(Xtrain, ytrain)
    print(f'{name:12s} R^2 = {r2_score(ytest, model.predict(Xtest)):.3f}')

**TODO**

1. Add more properties. Which ones are most strongly encoded?
2. If angular size or brightness dominates, repeat Section 3 using only galaxies in a
   narrow range of `D26`. Do the neighbors change?
3. Make a UMAP colored by each property (see the tutorial notebook).

## 6. Outliers

A galaxy whose nearest neighbors are all far away is unusual. We measure the similarity
to the *k*-th nearest neighbor for every galaxy and look at the lowest values. The
all-versus-all search is done once and cached.

In [ ]:
K = 10
knnfile = os.path.join(OUTDIR, f'knn-proj-{REGION}-k{K}.npz')

if os.path.isfile(knnfile):
    F = np.load(knnfile)
    assert np.array_equal(F['sgaid'], sgaid), 'Sample changed; delete the cached file.'
    knn_sims, knn_rows = F['sims'], F['rows']
else:
    knn_sims, knn_rows = rare.knn(proj, k=K)
    np.savez(knnfile, sgaid=sgaid, sims=knn_sims, rows=knn_rows)

kth_sim = knn_sims[:, -1]   # low = isolated in embedding space

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(kth_sim, bins=100)
ax.set_yscale('log')
ax.set_xlabel(f'Cosine similarity to neighbor {K}')
ax.set_ylabel('Number of galaxies')

outliers = np.argsort(kth_sim)[:50]
rare.show_cutout_grid(sgaid[outliers], cutout_index, ncols=10, figsize_per=1.5)

In [ ]:
rare.print_viewer_links(sgaid[outliers], sample, REGION)

**TODO:** sort the top few hundred outliers into classes (for example: real and
unusual, artifact, bright star, bad mosaic). What fraction is astrophysically
interesting?

## 7. Record visual inspection

Every by-eye classification goes into a CSV file, never just into a notebook cell.
`rare.record_vi` appends rows of `sgaid,label,date`.

In [ ]:
VIFILE = 'vi-rings.csv'   # keep this file in the git repository

# Example (uncomment and edit):
# rare.record_vi([1234567, 2345678], 'ring', VIFILE)
# rare.record_vi([3456789], 'not-ring', VIFILE)

## Next steps

- Cross-match `sample` to Galaxy Zoo DESI and to one or more ring catalogs, and save
  the matches to `OUTDIR`.
- Measure whether nearest neighbors share Galaxy Zoo labels (plan, Phase 2, step 3).
- Compare similarity search with a linear classifier trained on the embeddings (plan,
  Phase 2, step 4).